In [1]:
import os
os.getcwd()

'C:\\Users\\Drew\\Desktop\\cancer project\\programs\\scripts\\Project Code'

In [3]:
##Move jupyter to the directory of the scripts
import os
os.chdir(r'C:\Users\Drew\Desktop\cancer project\programs\scripts\Project Code')
os.getcwd()

'C:\\Users\\Drew\\Desktop\\cancer project\\programs\\scripts\\Project Code'

In [5]:
##tells how many cores 
import psutil
# Physical cores (actual hardware cores)
print(f"Physical CPU Cores: {psutil.cpu_count(logical=False)}")
# Logical processors (including hyper-threading)
print(f"Logical CPU Cores: {psutil.cpu_count(logical=True)}")
!nvidia-smi

Physical CPU Cores: 8
Logical CPU Cores: 16
Mon Apr 21 23:32:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 571.96                 Driver Version: 571.96         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080      WDDM  |   00000000:04:00.0  On |                  N/A |
| 53%   50C    P8             31W /  320W |     539MiB /  10240MiB |      3%      Default |
|                                         |                        |                  N/A |
+---

In [7]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from utils import *
import math
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from preprocessing import TumorPreprocessor
from discreterotation import DiscreteRotation
from ImageModel import ImageModel
import numpy as np
import random
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

In [11]:
#  _    _                                                            _                 
# | |  | |                                                          | |                
# | |__| |_   _ _ __   ___ _ __ _ __   __ _ _ __ __ _ _ __ ___   ___| |_ ___ _ __ ___  
# |  __  | | | | '_ \ / _ \ '__| '_ \ / _` | '__/ _` | '_ ` _ \ / _ \ __/ _ \ '__/ __| 
# | |  | | |_| | |_) |  __/ |  | |_) | (_| | | | (_| | | | | | |  __/ ||  __/ |  \__ \ 
# |_|  |_|\__, | .__/ \___|_|  | .__/ \__,_|_|  \__,_|_| |_| |_|\___|\__\___|_|  |___/ 
#          __/ | |             | |                                                     
#         |___/|_|             |_|            

totalIndeterminates = 1068
totalData = 401059
totalMalignants = 393
num_workers = 8
initial_learning_rate = 1e-4
weight_decay = 0.05
max_epochs = 20
patience_limit = 5
warmup_epochs = 2
#for cosine annealing
T_max = max_epochs - warmup_epochs
eta_min = 1e-6
batch_size = 32

malignant_weight = 100
labeling_strategy = "standard"
#labeling_strategy = "indeterminate_as_malignant"
if labeling_strategy == "indeterminate_as_malignant":
    undersamplingweight = totalIndeterminates/(totalData - totalIndeterminates)
else: #strategy is standard
    undersamplingweight = totalMalignants/(totalData-totalMalignants)
#these produce a 1:1 ratio of target:nontarget in sampling
#undersamplingweight = 0.001 can set this manually if you want to not do 1:1 ratio.

In [13]:
# ------------------------------------------------------------------------
# 1) Set seeds for reproducibility
# ------------------------------------------------------------------------
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [15]:
# ------------------------------------------------------------------------
# 2) Device setup
# ------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [17]:
# ------------------------------------------------------------------------
# 3) Data loaders
# ------------------------------------------------------------------------
#csv_file = r"C:\Users\Drew\Desktop\cancer project\train-metadata.csv"
csv_file = r"C:\Users\Drew\Desktop\cancer project\programs\scripts\Project Code\metadata_preprocessed.csv"
img_dir = r"C:\Users\Drew\Desktop\cancer project\train-image\image"

#old transforms
#these transforms for imagenet1k
#train_transform = transforms.Compose([
#    transforms.Resize((224, 224)),
#    transforms.RandomHorizontalFlip(p=0.5),
#    DiscreteRotation(),
#    transforms.RandomCrop(224, pad_if_needed=True),
#    transforms.ColorJitter(brightness=0.2, contrast=0.2),
#    transforms.ToTensor(),
#    transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                         std=[0.229, 0.224, 0.225])
#])

#these transforms for imagenet21k
#train_transform = transforms.Compose([
#    transforms.Resize((224, 224)),
#    transforms.RandomHorizontalFlip(p=0.5),
#    DiscreteRotation(),
    #transforms.RandomCrop(224, pad_if_needed=True),
    #transforms.ColorJitter(brightness=0.2, contrast=0.2),
#    transforms.ToTensor(),
#    transforms.Normalize(mean=[0.5, 0.5, 0.5],
#                         std=[0.5, 0.5, 0.5])
#])

#test_transform = transforms.Compose([
#    transforms.Resize((224, 224)),
#    transforms.ToTensor(),
#    transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                         std=[0.229, 0.224, 0.225])
#])

#test_transform = transforms.Compose([
#    transforms.Resize((224, 224)),
#    transforms.ToTensor(),
#    transforms.Normalize(mean=[0.5, 0.5, 0.5],
#                         std=[0.5, 0.5, 0.5])
#])


# Desired image size for training and validation
image_size = 224
#new transforms using albumentations
train_transform = A.Compose([
    A.Transpose(p=0.5),
    A.VerticalFlip(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.75),
    A.OneOf([
        A.MotionBlur(blur_limit=5),
        A.MedianBlur(blur_limit=5),
        A.GaussianBlur(blur_limit=5),
        A.GaussNoise(p=0.7)  # Removed var_limit to avoid warning
    ], p=0.7),
    A.OneOf([
        A.OpticalDistortion(distort_limit=1.0),
        A.GridDistortion(num_steps=5, distort_limit=1.0),
        A.ElasticTransform(alpha=3)
    ], p=0.7),
    A.CLAHE(clip_limit=4.0, p=0.7),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, border_mode=0, p=0.85),
    A.Resize(image_size, image_size),
    A.CoarseDropout(
        max_holes=1,
        max_height=int(image_size * 0.375),
        max_width=int(image_size * 0.375),
        fill_value=0,
        p=0.7
    ),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Define the validation transforms (usually just resize and normalize)
test_transform = A.Compose([
    A.Resize(image_size, image_size),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

preprocessor = TumorPreprocessor(
    csv_file, 
    img_dir, 
    train_transform, 
    test_transform, 
    batch_size=batch_size, 
    num_workers=num_workers,
    undersamplingweight = undersamplingweight,
    labeling_mode = labeling_strategy
)
train_loader, test_loader = preprocessor.get_dataloaders()

C:\Users\Drew\anaconda3\Lib\site-packages\albumentations\core\validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\Drew\AppData\Local\Temp\ipykernel_36736\2932940909.py:71: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(


In [18]:
# ------------------------------------------------------------------------
# 4) Construct and modify CNN model
# ------------------------------------------------------------------------

# Modify the final layer for binary classification. THIS IS FOR RESNET
#num_features = model.fc.in_features
#model.fc = nn.Linear(num_features, 1)  # 1 output for binary classification

model = timm.create_model('resnetv2_50x1_bitm', pretrained=True)
model.reset_classifier(num_classes=1)


#modify final layer, this is for efficientnet
#model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
#model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
#num_features = model.classifier[1].in_features

#model.classifier[1] = nn.Linear(num_features, 1)

#for ViT
#model = timm.create_model('vit_tiny_patch16_224.augreg_in21k_ft_in1k', pretrained=True)
#model = timm.create_model('vit_small_patch16_224.augreg_in21k_ft_in1k', pretrained=True)
#model = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True)
#convnext
#model = timm.create_model('convnextv2_tiny.fcmae_ft_in22k_in1k', pretrained=True)
# Get the number of input features from the current head
#in_features = model.head.in_features

# Replace the classification head inline with a sequential block

#model.head = nn.Sequential(
#    nn.Linear(in_features, 64),  # Reduce features to 64
#    nn.ReLU(inplace=True),       # Apply non-linearity
#    nn.Dropout(0.5),             # Dropout for regularization
#    nn.Linear(64, 1)             # Final layer for binary classification
#)

#for convnext
#model.reset_classifier(num_classes=1)


model.to(device)

C:\Users\Drew\anaconda3\Lib\site-packages\timm\models\_factory.py:126: UserWarning: Mapping deprecated model name resnetv2_50x1_bitm to current resnetv2_50x1_bit.goog_in21k_ft_in1k.
  model = create_fn(


ResNetV2(
  (stem): Sequential(
    (conv): StdConv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (pad): ConstantPad2d(padding=(1, 1, 1, 1), value=0.0)
    (pool): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (stages): Sequential(
    (0): ResNetStage(
      (blocks): Sequential(
        (0): PreActBottleneck(
          (downsample): DownsampleConv(
            (conv): StdConv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (norm): Identity()
          )
          (norm1): GroupNormAct(
            32, 64, eps=1e-05, affine=True
            (drop): Identity()
            (act): ReLU(inplace=True)
          )
          (conv1): StdConv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): GroupNormAct(
            32, 64, eps=1e-05, affine=True
            (drop): Identity()
            (act): ReLU(inplace=True)
          )
          (conv2): StdConv2d(64, 64, kernel_size=(3, 3)

In [19]:
# ------------------------------------------------------------------------
# 5) Create training components
# ------------------------------------------------------------------------
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([malignant_weight]).to(device))
optimizer = optim.AdamW(model.parameters(), lr=initial_learning_rate, weight_decay=weight_decay)
scaler = GradScaler()
# T_max is the number of iterations (epochs in this case) for one cycle
scheduler = CosineAnnealingLR(optimizer, T_max=T_max, eta_min=eta_min)

C:\Users\Drew\AppData\Local\Temp\ipykernel_36736\2208347356.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [25]:
# ------------------------------------------------------------------------
# 6) Wrap in ImageModel
# ------------------------------------------------------------------------
image_model = ImageModel(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scaler=scaler,
    device=device,
    test_loader=test_loader,
    model_name="vitsmallindeterminate"
)

In [27]:
# ------------------------------------------------------------------------
# 7) Training Loop with Manual Warmup, Cosine Annealing & Early Stopping
# ------------------------------------------------------------------------
best_metric = -1.0
patience_counter = 0
best_model_path = "vitsmall_cosine_warmup_best.pth"

print("Starting Training Loop with Warmup, Cosine Annealing, and Early Stopping...")

for epoch in range(max_epochs):
    print(f"\n===== Epoch {epoch+1}/{max_epochs} =====")

    # --- Train for one epoch ---
    image_model.train(train_loader, epochs=1, verbose=True) # Train for ONE epoch

    # --- Evaluate ---
    metrics = image_model.evaluate(verbose=True)
    val_loss = metrics.get('loss', float('inf'))
    val_pauc = metrics.get('pauc', -1.0)
    if np.isnan(val_pauc): val_pauc = -1.0
    current_metric = val_pauc

    print(f"Epoch {epoch+1}: Val Loss={val_loss:.4f}, Val pAUC={current_metric:.4f}")

    # --- LR Scheduling Step (Warmup + Cosine Annealing) ---
    current_lr = -1 # Initialize for logging
    if epoch < warmup_epochs:
        # Manual Linear Warmup: Adjust LR directly in the optimizer
        # Calculate the warmup factor (from 0 to 1 over warmup_epochs)
        # Note: This simple approach assumes stepping per epoch.
        # A more common approach steps per *batch*, which is more complex to add here
        # but provides smoother warmup, especially for short warmups.
        # Let's stick to epoch-level warmup for simplicity with this loop structure.
        # We need a starting LR for warmup, let's assume it starts near 0 or eta_min
        warmup_lr_start = eta_min # Start warmup from minimum LR
        lr_factor = (epoch + 1) / warmup_epochs # Goes from 1/warmup_epochs to 1.0
        current_warmup_lr = warmup_lr_start + lr_factor * (initial_learning_rate - warmup_lr_start)

        # Apply the calculated LR to all parameter groups in the optimizer
        for param_group in optimizer.param_groups:
            param_group['lr'] = current_warmup_lr
        current_lr = current_warmup_lr
        print(f"  Warmup Epoch {epoch+1}/{warmup_epochs}. Set LR to: {current_lr:.8f}")
        # DO NOT step the cosine scheduler during warmup
    else:
        # After warmup, step the CosineAnnealingLR scheduler
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0] # Get LR for logging
        print(f"  Cosine Annealing Phase. Stepped Scheduler. Current LR: {current_lr:.8f}")
    # --- End LR Scheduling ---

    # --- Early Stopping Check ---
    if current_metric > best_metric:
        best_metric = current_metric
        print(f"  Metric improved to {best_metric:.4f}. Saving model...")
        image_model.save_model(best_model_path)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  Metric did not improve. Best: {best_metric:.4f}. Patience: {patience_counter}/{patience_limit}")

    if patience_counter >= patience_limit:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

print("\nTraining finished.")
# --- Load Best Model ---
print(f"Loading best model from {best_model_path} with metric: {best_metric:.4f}")
image_model.load_model(best_model_path)

# --- Final Evaluation ---
print("\nFinal evaluation using the best model:")
final_metrics = image_model.evaluate(verbose=True)

Starting Training Loop with Warmup, Cosine Annealing, and Early Stopping...

===== Epoch 1/20 =====


C:\Users\Drew\Desktop\cancer project\programs\scripts\Project Code\ImageModel.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/1], Step [50/10027], Loss: 2.8589
Epoch [1/1], Step [100/10027], Loss: 2.4147
Epoch [1/1], Step [150/10027], Loss: 2.0015
Epoch [1/1], Step [200/10027], Loss: 2.7502
Epoch [1/1], Step [250/10027], Loss: 2.5550
Epoch [1/1], Step [300/10027], Loss: 2.7437
Epoch [1/1], Step [350/10027], Loss: 2.1166
Epoch [1/1], Step [400/10027], Loss: 2.3262
Epoch [1/1], Step [450/10027], Loss: 1.6838
Epoch [1/1], Step [500/10027], Loss: 4.2182
Epoch [1/1], Step [550/10027], Loss: 2.2792
Epoch [1/1], Step [600/10027], Loss: 2.2489
Epoch [1/1], Step [650/10027], Loss: 2.8827
Epoch [1/1], Step [700/10027], Loss: 2.5148
Epoch [1/1], Step [750/10027], Loss: 1.5636
Epoch [1/1], Step [800/10027], Loss: 2.5557
Epoch [1/1], Step [850/10027], Loss: 2.4720
Epoch [1/1], Step [900/10027], Loss: 1.8097
Epoch [1/1], Step [950/10027], Loss: 2.3121
Epoch [1/1], Step [1000/10027], Loss: 3.2070
Epoch [1/1], Step [1050/10027], Loss: 2.0339
Epoch [1/1], Step [1100/10027], Loss: 2.7372
Epoch [1/1], Step [1150/10027]

C:\Users\Drew\Desktop\cancer project\programs\scripts\Project Code\ImageModel.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/1], Step [50/10027], Loss: 2.2173
Epoch [1/1], Step [100/10027], Loss: 1.7568
Epoch [1/1], Step [150/10027], Loss: 2.1174
Epoch [1/1], Step [200/10027], Loss: 1.9464
Epoch [1/1], Step [250/10027], Loss: 2.4215
Epoch [1/1], Step [300/10027], Loss: 2.4062
Epoch [1/1], Step [350/10027], Loss: 1.4552
Epoch [1/1], Step [400/10027], Loss: 2.4785
Epoch [1/1], Step [450/10027], Loss: 2.3256
Epoch [1/1], Step [500/10027], Loss: 1.4959
Epoch [1/1], Step [550/10027], Loss: 1.8517
Epoch [1/1], Step [600/10027], Loss: 2.4687
Epoch [1/1], Step [650/10027], Loss: 2.1218
Epoch [1/1], Step [700/10027], Loss: 1.6776
Epoch [1/1], Step [750/10027], Loss: 2.4056
Epoch [1/1], Step [800/10027], Loss: 1.8396
Epoch [1/1], Step [850/10027], Loss: 2.0816
Epoch [1/1], Step [900/10027], Loss: 1.9468
Epoch [1/1], Step [950/10027], Loss: 1.8956
Epoch [1/1], Step [1000/10027], Loss: 1.6184
Epoch [1/1], Step [1050/10027], Loss: 1.4814
Epoch [1/1], Step [1100/10027], Loss: 1.7057
Epoch [1/1], Step [1150/10027]

...

Epoch [1/1], Step [50/10027], Loss: 0.5131
Epoch [1/1], Step [100/10027], Loss: 0.6283
Epoch [1/1], Step [150/10027], Loss: 0.3964
Epoch [1/1], Step [200/10027], Loss: 1.2177
Epoch [1/1], Step [250/10027], Loss: 1.6908
Epoch [1/1], Step [300/10027], Loss: 0.9059
Epoch [1/1], Step [350/10027], Loss: 1.0720
Epoch [1/1], Step [400/10027], Loss: 0.8892
Epoch [1/1], Step [450/10027], Loss: 0.3030
Epoch [1/1], Step [500/10027], Loss: 1.1003
Epoch [1/1], Step [550/10027], Loss: 0.8725
Epoch [1/1], Step [600/10027], Loss: 0.6912
Epoch [1/1], Step [650/10027], Loss: 1.0294
Epoch [1/1], Step [700/10027], Loss: 1.2412
Epoch [1/1], Step [750/10027], Loss: 0.7852
Epoch [1/1], Step [800/10027], Loss: 0.5434
Epoch [1/1], Step [850/10027], Loss: 1.3041
Epoch [1/1], Step [900/10027], Loss: 0.9470
Epoch [1/1], Step [950/10027], Loss: 0.3525
Epoch [1/1], Step [1000/10027], Loss: 1.0907
Epoch [1/1], Step [1050/10027], Loss: 0.9015
Epoch [1/1], Step [1100/10027], Loss: 0.6854
Epoch [1/1], Step [1150/10027]

C:\Users\Drew\Desktop\cancer project\programs\scripts\Project Code\ImageModel.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/1], Step [50/10027], Loss: 1.1185
Epoch [1/1], Step [100/10027], Loss: 0.6367
Epoch [1/1], Step [150/10027], Loss: 0.9266
Epoch [1/1], Step [200/10027], Loss: 0.6563
Epoch [1/1], Step [250/10027], Loss: 0.6182
Epoch [1/1], Step [300/10027], Loss: 0.7626
Epoch [1/1], Step [350/10027], Loss: 0.7925
Epoch [1/1], Step [400/10027], Loss: 0.4467
Epoch [1/1], Step [450/10027], Loss: 0.5088
Epoch [1/1], Step [500/10027], Loss: 0.9015
Epoch [1/1], Step [550/10027], Loss: 0.7556
Epoch [1/1], Step [600/10027], Loss: 0.9603
Epoch [1/1], Step [650/10027], Loss: 0.5686
Epoch [1/1], Step [700/10027], Loss: 1.1061
Epoch [1/1], Step [750/10027], Loss: 0.6134
Epoch [1/1], Step [800/10027], Loss: 0.5061
Epoch [1/1], Step [850/10027], Loss: 0.8984
Epoch [1/1], Step [900/10027], Loss: 0.9442
Epoch [1/1], Step [950/10027], Loss: 0.3136
Epoch [1/1], Step [1000/10027], Loss: 2.0474
Epoch [1/1], Step [1050/10027], Loss: 1.2837
Epoch [1/1], Step [1100/10027], Loss: 0.8656
Epoch [1/1], Step [1150/10027]

C:\Users\Drew\Desktop\cancer project\programs\scripts\Project Code\ImageModel.py:227: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filepath, map_loc


Final evaluation using the best model:
Test Loss: 0.9532, Test Accuracy: 77.53%
True Positives: 74
False Negatives: 5
False Positives: 18019
True Negatives: 62114
Sensitivity (TPR): 0.9367
Specificity (TNR): 0.7751
pAUC (min_tpr=0.80): 0.1556


In [ ]:
# ------------------------------------------------------------------------
# 7) Train & Evaluate
# ------------------------------------------------------------------------
image_model.train(train_loader, epochs=1, verbose=True)

In [ ]:
metrics = image_model.evaluate(verbose=True)

In [51]:
# ------------------------------------------------------------------------
# 8) Save the trained model checkpoint
# ------------------------------------------------------------------------
save_path = r"C:\Users\Drew\Desktop\cancer project\programs\scripts\Project Code\models\EffNetb3-1mweight1epochSTANDARD.pth"
image_model.save_model(save_path)
print(f"Model saved to {save_path}")

Model saved to C:\Users\Drew\Desktop\cancer project\programs\scripts\Project Code\models\EffNetb3-1mweight1epochSTANDARD.pth
